In [21]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR, SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, accuracy_score, classification_report
import pickle

# ============ STEP 1 - LOAD DATA ============
df = pd.read_csv(r"C:\Users\prade\Downloads\archive (3)\BollywoodMovieDetail.csv")
print("Shape:", df.shape)

# ============ STEP 2 - CLEAN DATA ============
df.dropna(subset=['genre', 'actors', 'directors', 'hitFlop'], inplace=True)
df.drop_duplicates(inplace=True)
df = df.fillna(0)
df.reset_index(drop=True, inplace=True)
print("Clean Shape:", df.shape)

# ============ STEP 3 - FEATURE ENGINEERING ============
df['primary_genre'] = df['genre'].apply(lambda x: x.split('|')[0].strip())
df['primary_actor'] = df['actors'].apply(lambda x: x.split('|')[0].strip())
df['primary_director'] = df['directors'].apply(lambda x: x.split('|')[0].strip())

# HitFlop simplify
def simplify_hit(x):
    if x >= 6:
        return 'Hit'
    elif x >= 3:
        return 'Average'
    else:
        return 'Flop'
df['hitFlop_label'] = df['hitFlop'].apply(simplify_hit)

# ============ STEP 4 - ENCODING ============
le_genre = LabelEncoder()
le_actor = LabelEncoder()
le_director = LabelEncoder()

df['genre_enc'] = le_genre.fit_transform(df['primary_genre'])
df['actor_enc'] = le_actor.fit_transform(df['primary_actor'])
df['director_enc'] = le_director.fit_transform(df['primary_director'])

print("\nSample:")
print(df[['title', 'primary_genre', 'primary_actor', 'hitFlop', 'hitFlop_label']].head())

# ============ STEP 5 - FEATURES & TARGET ============
X = df[['genre_enc', 'actor_enc', 'director_enc', 'releaseYear', 'sequel']]
y_reg = df['hitFlop'].astype(float)
y_clf = df['hitFlop_label']

# ============ STEP 6 - TRAIN TEST SPLIT ============
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

# ============ STEP 7 - SCALING ============
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ============ STEP 8 - SVR ============
print("\n--- SVR Regressor ---")
svr = SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale')
svr.fit(X_train_s, y_train_reg)

print("Training R2:", r2_score(y_train_reg, svr.predict(X_train_s)))
print("Testing R2:", r2_score(y_test_reg, svr.predict(X_test_s)))

# ============ STEP 9 - SVC ============
print("\n--- SVC Classifier ---")
svc = SVC(kernel='rbf', C=100, gamma='scale', random_state=42)
svc.fit(X_train_s, y_train_clf)

print("Training Accuracy:", accuracy_score(y_train_clf, svc.predict(X_train_s)))
print("Testing Accuracy:", accuracy_score(y_test_clf, svc.predict(X_test_s)))
print(classification_report(y_test_clf, svc.predict(X_test_s)))

# ============ STEP 10 - SAVE ============
pickle.dump(svr, open('svr_model.pkl', 'wb'))
pickle.dump(svc, open('svc_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler_movie.pkl', 'wb'))
pickle.dump(le_genre, open('le_genre.pkl', 'wb'))
pickle.dump(le_actor, open('le_actor.pkl', 'wb'))
pickle.dump(le_director, open('le_director.pkl', 'wb'))
df.to_csv('bollywood_clean.csv', index=False)
print("\nSab save ho gaya!")

# ============ STEP 11 - RECOMMENDATION ============
def recommend_movies(genre=None, actor=None, director=None, top_n=5):
    filtered = df.copy()

    if genre:
        filtered = filtered[filtered['genre'].str.contains(genre, case=False, na=False)]
    if actor:
        filtered = filtered[filtered['actors'].str.contains(actor, case=False, na=False)]
    if director:
        filtered = filtered[filtered['directors'].str.contains(director, case=False, na=False)]

    if filtered.empty:
        print("Koi movie nahi mili is filter se!")
        return None

    features = filtered[['genre_enc', 'actor_enc', 'director_enc', 'releaseYear', 'sequel']]
    features_scaled = scaler.transform(features)
    filtered = filtered.copy()
    filtered['predicted_score'] = svr.predict(features_scaled)
    filtered['predicted_label'] = svc.predict(features_scaled)

    result = filtered[['title', 'primary_genre', 'primary_actor', 
                       'releaseYear', 'hitFlop', 'predicted_score', 
                       'predicted_label']]
    result = result.sort_values('predicted_score', ascending=False).head(top_n)

    print(f"\n🎬 Top {top_n} Recommended Movies:")
    print(result.to_string(index=False))
    return result

# ============ STEP 12 - TEST ============
print("\n=== Genre: Romance ===")
recommend_movies(genre='Romance', top_n=5)

print("\n=== Actor: Shah Rukh Khan ===")
recommend_movies(actor='Shah Rukh Khan', top_n=5)

print("\n=== Director: Karan Johar ===")
recommend_movies(director='Karan Johar', top_n=5)

Shape: (1275, 17)
Clean Shape: (1275, 17)

Sample:
                               title primary_genre   primary_actor  hitFlop  \
0                             Albela       Romance         Govinda        2   
1  Lagaan: Once Upon a Time in India     Adventure      Aamir Khan        6   
2           Meri Biwi Ka Jawab Nahin        Action    Akshay Kumar        1   
3             Hum Tumhare Hain Sanam         Drama  Shah Rukh Khan        4   
4                         One 2 Ka 4        Action  Shah Rukh Khan        1   

  hitFlop_label  
0          Flop  
1           Hit  
2          Flop  
3       Average  
4          Flop  

--- SVR Regressor ---
Training R2: 0.16875237169405477
Testing R2: -0.1380298470087269

--- SVC Classifier ---
Training Accuracy: 0.8009803921568628
Testing Accuracy: 0.7725490196078432
              precision    recall  f1-score   support

     Average       0.50      0.05      0.09        41
        Flop       0.78      0.99      0.87       195
         Hit    

,title,primary_genre,primary_actor,releaseYear,hitFlop,predicted_score,predicted_label
7,Kabhi Khushi Kabhie Gham...,Drama,Amitabh Bachchan,2001,8,2.573597,Flop
1011,Student of the Year,Comedy,Sidharth Malhotra,2012,5,1.284416,Flop
622,My Name Is Khan,Drama,Shah Rukh Khan,2010,6,1.225065,Flop
353,Kabhi Alvida Naa Kehna,Drama,Shah Rukh Khan,2006,5,1.207863,Flop
1161,Bombay Talkies,Drama,Rani Mukerji,2013,2,1.198801,Flop


In [19]:
df.to_csv(r"C:\Users\prade\Downloads\archive (3)\BollywoodMovieDetail.csv", index=False)
print(df.columns.tolist())

['imdbId', 'title', 'releaseYear', 'releaseDate', 'genre', 'writers', 'actors', 'directors', 'sequel', 'hitFlop', 'primary_genre', 'primary_actor', 'primary_director', 'hitFlop_label', 'genre_enc', 'actor_enc', 'director_enc']
